<a href="https://colab.research.google.com/github/TManhNguyen/OneClick-PrettyMaps/blob/main/PrettyMaps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OneClick PrettyMaps 🗺️

Colourful map posters of any place, in four steps. Run the cells top to bottom with the ▶ buttons.

1. **Setup** – installs everything (once per session, ~1–2 min).
2. **Choose the area** – search or click on the map, pick circle / square, or draw your own shape.
3. **Download** – fetches OpenStreetMap data for that area.
4. **Style it** – pick a palette (or paste any [Color Hunt](https://colorhunt.co) link) and effects. Re-run step 4 as often as you like; nothing is downloaded again.

Built on [prettymaps](https://github.com/marceloprates/prettymaps) by **Marcelo Prates** · Map data © [OpenStreetMap](https://www.openstreetmap.org/copyright) contributors · Licensed AGPL-3.0

In [ ]:
#@title 1 · Setup (run once)
Version = "main" #@param {type:"string"}
!pip install -q "oneclick-prettymaps[picker] @ git+https://github.com/TManhNguyen/OneClick-PrettyMaps@{Version}"
!pip install -q --force-reinstall --no-deps "oneclick-prettymaps @ git+https://github.com/TManhNguyen/OneClick-PrettyMaps@{Version}"
# prettymaps goes last and without its own dependency list: that list upgrades
# ipykernel (breaks Colab) and pulls ~400 MB of pen-plotter packages we don't use.
# Installing it last also keeps pip from printing warnings about that list.
!pip install -q --no-deps "prettymaps @ git+https://github.com/marceloprates/prettymaps@02f85870ced807b7d24ce1764764ff877f9edffd"

from google.colab import output, files
output.enable_custom_widget_manager()

import sys
# Forget any copy imported earlier in this runtime, so the version just installed is used
for name in [m for m in sys.modules if m.split(".")[0] == "oneclick_prettymaps"]:
    del sys.modules[name]

import matplotlib.pyplot as plt
import oneclick_prettymaps as opm

print("Fonts:", opm.load_fonts())
print(f"oneclick_prettymaps {opm.__version__} ready ✔  (needs 0.3.0+)")

In [ ]:
#@title 2 · Choose the area
#@markdown Search with the 🔍 on the map, click to move the pin, then pick a shape and radius.
#@markdown Choose **Drawn shape** to draw any polygon or rectangle instead.
#@markdown If the pin looks stuck in a corner, press **Recentre**.
Start_at = "Ho Chi Minh City" #@param {type:"string"}

picker = opm.MapPicker(Start_at, radius=1000).show()

In [ ]:
#@title 3 · Download map data
#@markdown Leave **Type_location_instead** empty to use the map above.
#@markdown If the map does not show, type a place or `lat, lon` here.
Type_location_instead = "" #@param {type:"string"}
Radius_meters = 1000 #@param {type:"slider", min:100, max:8000, step:50}
Circle_map = True #@param {type:"boolean"}
#@markdown Sea is slower to compute — tick it for coastal places only.
Include_sea = False #@param {type:"boolean"}

import re
if Type_location_instead.strip():
    m = re.fullmatch(r"\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*", Type_location_instead)
    query = (float(m[1]), float(m[2])) if m else Type_location_instead
    selection = dict(query=query, radius=Radius_meters, circle=Circle_map)
else:
    selection = picker.selection()

gdfs = opm.fetch(**selection, sea=Include_sea)
print({k: len(v) for k, v in gdfs.items()})

In [ ]:
#@title 4 · Style it (re-run freely — no re-download)
#@markdown **Palette**: pick one, or paste a Color Hunt link like `https://colorhunt.co/palette/f9ed69f08a5db83b5e6a2c70`
Palette = "saigon-sunset" #@param ["saigon-sunset", "pastel-pop", "tropical", "terracotta", "nordic", "mint-candy", "retro-80s", "autumn", "ink", "neon-night", "blueprint"] {allow-input: true}
Mode = "auto" #@param ["auto", "light", "dark"]
#@markdown **Buildings** – `auto` uses building types when the city has them tagged, otherwise size.
Buildings = "auto" #@param ["auto", "type", "height", "size", "distance", "random"]
#@markdown **Streets** – `orientation` = rainbow by compass direction.
Streets = "type" #@param ["type", "orientation", "solid"]
Glow = "auto" #@param ["auto", "on", "off"]
Shadows = False #@param {type:"boolean"}
Hatch = False #@param {type:"boolean"}
Grain = 0.05 #@param {type:"slider", min:0, max:0.15, step:0.01}
#@markdown **Land use** tints residential / commercial / industrial areas — brings colour where few buildings are mapped.
Land_use = 0.35 #@param {type:"slider", min:0, max:0.7, step:0.05}
#@markdown ---
Title = "Sai Gon" #@param {type:"string"}
Subtitle = "auto" #@param {type:"string"}
Legend = False #@param {type:"boolean"}
Layout = "poster" #@param ["poster", "square", "a4", "wallpaper"]
Seed = 0 #@param {type:"integer"}
#@markdown ---
Download_PNG = False #@param {type:"boolean"}
Download_SVG = False #@param {type:"boolean"}
DPI = 300 #@param {type:"slider", min:100, max:600, step:50}

fig = opm.render(
    gdfs,
    theme=Palette,
    dark=None if Mode == "auto" else Mode == "dark",
    glow=None if Glow == "auto" else Glow == "on",
    buildings=Buildings,
    streets=Streets,
    shadows=Shadows,
    hatch=Hatch,
    grain=Grain,
    landuse=Land_use,
    title=Title or None,
    subtitle=Subtitle or None,
    legend=Legend,
    layout=Layout,
    seed=Seed,
)
plt.show()

name = re.sub(r"\W+", "_", Title or "map").strip("_").lower() or "map"
if Download_PNG:
    files.download(opm.save(fig, f"{name}.png", dpi=DPI))
if Download_SVG:
    files.download(opm.save(fig, f"{name}.svg"))

In [ ]:
#@title 5 · Palette gallery (optional) — preview every palette on your map
#@markdown Can take a minute for big areas.
Gallery_streets = "orientation" #@param ["type", "orientation", "solid"]

for p in opm.PALETTES:
    f = opm.render(gdfs, theme=p, streets=Gallery_streets, title=p, subtitle=None,
                   layout="square", figsize=(4, 4))
    plt.show()